# mfs-agent-security: benchmark run on Kaggle

**Before running:** in the right panel, set **Accelerator: GPU T4 x2** and turn **Internet on**.

This version tests three models of different sizes, each with no defence and with the provenance defence, 3 times each. Expect about 8 to 9 hours in total, so use **Save Version → Save & Run All (Commit)** and let it run in the background. Results are saved after every run, so a stopped session still keeps what finished.

In [ ]:
!nvidia-smi

## 1. Get the project and run its tests

In [ ]:
!git clone -q https://github.com/s4m404/mfs-agent-security.git
%cd mfs-agent-security
!pip install -q -r requirements.txt
!python -m pytest -q

## 2. Install vLLM

In [ ]:
!pip install -q vllm

## 3. Settings
Edit these lists to change what runs. Each model runs once per defence.

In [ ]:
MODELS = [
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen2.5-14B-Instruct-AWQ",   # larger model, 4-bit quantised to fit the T4s
    "Qwen/Qwen2.5-3B-Instruct",        # smaller model
]
DEFENCES = ["none", "provenance"]
REPEATS = 3

## 4. Helpers: start and stop the model server

In [ ]:
import subprocess, time, urllib.request, json, glob, os, signal
from openai import OpenAI

def start_server(model):
    proc = subprocess.Popen(
        ["vllm", "serve", model,
         "--dtype", "half",                 # T4 GPUs do not support bfloat16
         "--tensor-parallel-size", "2",
         "--max-model-len", "8192",
         "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
         "--port", "8000"],
        stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for i in range(120):  # up to 20 minutes
        try:
            urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
            print(f"{model}: server ready after about {i * 10} seconds")
            return proc
        except Exception:
            if proc.poll() is not None:
                break
            time.sleep(10)
    print(f"{model}: server did not start. Last lines of vllm.log:")
    print(open("vllm.log").read()[-3000:])
    stop_server(proc)
    return None

def stop_server(proc):
    if proc and proc.poll() is None:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        try:
            proc.wait(timeout=60)
        except Exception:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
    time.sleep(15)  # let the GPUs free their memory

def tool_call_check(model):
    client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
    resp = client.chat.completions.create(
        model=model, temperature=0,
        messages=[{"role": "user", "content": "What is my wallet balance?"}],
        tools=[{"type": "function", "function": {
            "name": "check_balance", "description": "Return the user's balance.",
            "parameters": {"type": "object", "properties": {}}}}])
    ok = bool(resp.choices[0].message.tool_calls)
    print(f"{model}: tool calling {'works' if ok else 'DOES NOT WORK, skipping this model'}")
    return ok

## 5. Run everything
Each model: start server, check tool calling, run every defence, stop server.

In [ ]:
for model in MODELS:
    print(f"\n########## {model} ##########")
    proc = start_server(model)
    if proc is None:
        continue
    try:
        if not tool_call_check(model):
            continue
        for defence in DEFENCES:
            print(f"\n===== {model} | defence={defence} =====")
            !python scripts/run_bench.py --model {model} --base-url http://localhost:8000/v1 --defence {defence} --repeats {REPEATS} 2>&1 | grep -v "HTTP Request" | tail -25
            !zip -qr /kaggle/working/results.zip results
    finally:
        stop_server(proc)

## 6. Summary of all runs
`results.zip` in the Output panel has everything. Download it and send it to Claude.

In [ ]:
!zip -qr /kaggle/working/results.zip results
for p in sorted(glob.glob("results/*/summary.md")):
    print(open(p).read().split("## Attack success by injection language")[0])